# Segmentasi Customer Online Retail II dengan RFM dan KMeans

**Dataset:** `online_retail_II.xlsx` — 2 sheet, Desember 2009 sampai Desember 2011

**Sampel:** 1.000 customer acak, diambil setelah cleaning

**Periode:** gabungan 2009–2011

**Retur/pembatalan:** dikeluarkan dari perhitungan RFM

**Snapshot RFM:** tanggal transaksi maksimum + 1 hari

Notebook ini melakukan audit data, cleaning, sampling per customer, perhitungan RFM, preprocessing KMeans, pemilihan `k`, profiling segmen, dan validasi stabilitas.

## 1. Setup

Semua path, parameter, dan library didefinisikan di satu tempat agar mudah direproduksi.

In [1]:
from pathlib import Path
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", palette="deep")

BASE = Path("/Users/march/Documents/MATKUL/PSD")  # folder proyek PSD — semua output ditulis di sini
SRC = Path("/Users/march/Downloads/online_retail_II.xlsx")  # file mentah — ganti di sini saat dataset diperbarui
DATA = BASE / "data"
REPORTS = BASE / "reports"
FIGS = BASE / "figures"
for folder in (DATA, REPORTS, FIGS):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CUSTOMERS = 1000
K_MIN, K_MAX = 2, 10

print("Setup selesai.")
print("Sumber data:", SRC)

Setup selesai.
Sumber data: /Users/march/Downloads/online_retail_II.xlsx


## 2. Load data

Kedua sheet dibaca dengan engine `calamine` agar pemrosesan file ±45 MB lebih cepat. Kolom `SourceSheet` ditambahkan untuk melacak asal periode.

In [2]:
t0 = time.time()
raw_sheets = pd.read_excel(SRC, sheet_name=None, engine="calamine")
raw = pd.concat(
    [frame.assign(SourceSheet=name) for name, frame in raw_sheets.items()],
    ignore_index=True,
)
print(f"Data dimuat: {len(raw):,} baris dalam {time.time() - t0:.1f} detik")
print("Sheet:", list(raw_sheets))
display(raw.head())
display(raw.dtypes.to_frame("dtype"))
display(raw.isna().sum().to_frame("missing"))

Data dimuat: 1,067,371 baris dalam 8.2 detik
Sheet: ['Year 2009-2010', 'Year 2010-2011']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


,dtype
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str
SourceSheet,str


,missing
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0
SourceSheet,0


## 3. Cleaning

Aturan yang dipakai:

1. Menyeragamkan tipe dan format teks.
2. Menghapus duplikat persis.
3. Mengeluarkan invoice pembatalan (`C`) dan adjustment (`A`).
4. Mengeluarkan `Quantity <= 0` dan `UnitPrice <= 0`.
5. Mengeluarkan customer tanpa `CustomerID`.
6. Mengeluarkan StockCode non-produk (ongkir, manual, fee, voucher, tes).

StockCode **tidak** difilter berdasarkan panjang 5 digit karena produk valid juga bisa 6–9 karakter, misalnya `79323LP`, `15056BL`, dan `DCGS0058`.

In [3]:
df = raw.rename(columns={"Price": "UnitPrice"}).copy()

df["Invoice"] = df["Invoice"].astype(str).str.strip().str.removesuffix(".0")
df["StockCode"] = (
    df["StockCode"].astype(str).str.strip().str.upper().str.removesuffix(".0")
)
df["Description"] = df["Description"].astype("string").str.strip()
df["CustomerID"] = df["Customer ID"].astype("Float64").astype("Int64").astype("string")
df = df.drop(columns=["Customer ID"])
df["Country"] = df["Country"].astype("string").str.strip()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

df = df.dropna(subset=["InvoiceDate", "Quantity", "UnitPrice"]).copy()
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

ORIGINAL_COLUMNS = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]

NON_PRODUCT_CODES = {
    "POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES",
    "ADJUST", "ADJUST2", "AMAZONFEE", "PADS", "CRUK",
    "B", "TEST001", "TEST002", "DCGSSGIRL", "DCGSSBOY",
}

funnel = []


def record_step(name, before, after):
    funnel.append({
        "step": name,
        "before": before,
        "removed": before - after,
        "remaining": after,
    })


record_step("Data awal (setelah parsing)", len(df), len(df))

before = len(df)
df = df.drop_duplicates(subset=ORIGINAL_COLUMNS).copy()
record_step("Duplikat persis", before, len(df))

before = len(df)
df = df[~df["Invoice"].str.upper().str[0].isin(["C", "A"])].copy()
record_step("Invoice C/A", before, len(df))

before = len(df)
df = df[df["Quantity"] > 0].copy()
record_step("Quantity <= 0", before, len(df))

before = len(df)
df = df[df["UnitPrice"] > 0].copy()
record_step("UnitPrice <= 0", before, len(df))

before = len(df)
df = df[df["CustomerID"].notna()].copy()
record_step("CustomerID kosong", before, len(df))

before = len(df)
non_product = df["StockCode"].isin(NON_PRODUCT_CODES) | df["StockCode"].str.startswith("GIFT_")
df = df[~non_product].copy()
record_step("StockCode non-produk", before, len(df))

df = df.reset_index(drop=True)

print(f"Baris bersih: {len(df):,}")
print(f"Customer unik: {df['CustomerID'].nunique():,}")
print(f"Invoice unik: {df['Invoice'].nunique():,}")
print(f"Rentang tanggal: {df['InvoiceDate'].min()} s.d. {df['InvoiceDate'].max()}")

Baris bersih: 776,579
Customer unik: 5,852
Invoice unik: 36,594
Rentang tanggal: 2009-12-01 07:45:00 s.d. 2011-12-09 12:50:00


In [4]:
funnel_df = pd.DataFrame(funnel)
display(funnel_df)
funnel_df.to_csv(REPORTS / "cleaning_funnel.csv", index=False)

print("Contoh transaksi bersih:")
display(df.head(10))

,step,before,removed,remaining
0,Data awal (setelah parsing),1067371,0,1067371
1,Duplikat persis,1067371,34335,1033036
2,Invoice C/A,1033036,19110,1013926
3,Quantity <= 0,1013926,3393,1010533
4,UnitPrice <= 0,1010533,2621,1007912
5,CustomerID kosong,1007912,228487,779425
6,StockCode non-produk,779425,2846,776579


Contoh transaksi bersih:


,Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country,SourceSheet,CustomerID,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,United Kingdom,Year 2009-2010,13085,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom,Year 2009-2010,13085,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,United Kingdom,Year 2009-2010,13085,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,United Kingdom,Year 2009-2010,13085,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,United Kingdom,Year 2009-2010,13085,30.0
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,United Kingdom,Year 2009-2010,13085,39.6
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,United Kingdom,Year 2009-2010,13085,30.0
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,United Kingdom,Year 2009-2010,13085,59.5
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,United Kingdom,Year 2009-2010,13085,30.6
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,United Kingdom,Year 2009-2010,13085,45.0


## 4. Sampling 1.000 customer

Sampling dilakukan **per customer**, bukan per invoice. Tujuannya agar seluruh riwayat belanja customer terpilih tetap utuh sehingga Recency, Frequency, dan Monetary valid.

Data dibersihkan terlebih dahulu, baru kemudian customer dipilih secara acak dengan seed tetap.

In [5]:
customers = np.array(sorted(df["CustomerID"].dropna().unique()))
rng = np.random.default_rng(SEED)
sampled_customers = rng.choice(customers, size=N_CUSTOMERS, replace=False)
sample = df[df["CustomerID"].isin(sampled_customers)].copy().reset_index(drop=True)

sample_info = pd.DataFrame({
    "metric": [
        "customer_populasi", "customer_sampel",
        "baris_populasi", "baris_sampel",
        "invoice_sampel", "tanggal_awal_sampel", "tanggal_akhir_sampel",
        "revenue_sampel_gbp",
    ],
    "value": [
        len(customers), sample["CustomerID"].nunique(),
        len(df), len(sample),
        sample["Invoice"].nunique(),
        sample["InvoiceDate"].min(), sample["InvoiceDate"].max(),
        round(sample["Revenue"].sum(), 2),
    ],
})
display(sample_info)

sample.to_csv(DATA / "clean_transactions_sample.csv", index=False)
print(f"Transaksi sampel disimpan: {len(sample):,} baris")

,metric,value
0,customer_populasi,5852
1,customer_sampel,1000
2,baris_populasi,776579
3,baris_sampel,126269
4,invoice_sampel,6544
5,tanggal_awal_sampel,2009-12-01 09:24:00
6,tanggal_akhir_sampel,2011-12-09 11:50:00
7,revenue_sampel_gbp,3374321.4


Transaksi sampel disimpan: 126,269 baris


## 5. Perhitungan RFM

Didefinisikan:

```text
R = (snapshot - tanggal transaksi terakhir customer).days
F = jumlah Invoice unik
M = sum(Quantity * UnitPrice)
```

Snapshot memakai tanggal maksimum dataset penuh + 1 hari agar Recency tidak bias oleh sampel. RFM score 1–5 dibuat dengan `qcut` untuk label bisnis; nilai R/F/M asli tetap dipakai untuk KMeans setelah transformasi.

In [6]:
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot RFM:", snapshot)


def build_rfm(frame, snapshot):
    out = frame.groupby("CustomerID").agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum"),
        Country=("Country", lambda s: s.mode().iloc[0] if len(s.mode()) else pd.NA),
    )
    out["Recency"] = (snapshot - out["LastPurchase"]).dt.days
    return out.drop(columns="LastPurchase").reset_index()


rfm_full = build_rfm(df, snapshot)
rfm = build_rfm(sample, snapshot)
rfm = rfm[["CustomerID", "Recency", "Frequency", "Monetary", "Country"]]

# RFM score: Recency kecil = lebih baik (score 5), Frequency/Monetary besar = score 5
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first", ascending=False),
    5, labels=[1, 2, 3, 4, 5],
).astype(int)
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first", ascending=True),
    5, labels=[1, 2, 3, 4, 5],
).astype(int)
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first", ascending=True),
    5, labels=[1, 2, 3, 4, 5],
).astype(int)
rfm["RFM_Score"] = rfm[["R_Score", "F_Score", "M_Score"]].sum(axis=1)
rfm["RFM_Cell"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

print("Ringkasan RFM sampel:")
display(rfm[["Recency", "Frequency", "Monetary"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
).T)

summary_rows = []
for label, frame in [("populasi", rfm_full), ("sampel", rfm)]:
    summary_rows.append({
        "dataset": label,
        "customers": len(frame),
        "recency_median": frame["Recency"].median(),
        "frequency_median": frame["Frequency"].median(),
        "monetary_median": round(frame["Monetary"].median(), 2),
        "monetary_mean": round(frame["Monetary"].mean(), 2),
        "monetary_total": round(frame["Monetary"].sum(), 2),
    })
rfm_summary = pd.DataFrame(summary_rows)
display(rfm_summary)
rfm_summary.to_csv(REPORTS / "rfm_summary.csv", index=False)
rfm.to_csv(DATA / "rfm_sample.csv", index=False)

Snapshot RFM: 2011-12-10 12:50:00


Ringkasan RFM sampel:


,count,mean,std,min,25%,50%,75%,95%,99%,max
Recency,1000.0,201.5590,208.680713,1.00,25.0000,92.00,380.0000,617.0500,729.0100,739.00
Frequency,1000.0,6.5440,16.048727,1.00,1.0000,3.00,7.0000,20.0000,52.0100,373.00
Monetary,1000.0,3374.3214,22426.726359,2.95,333.7925,892.16,2154.0525,7572.1295,32280.0334,580987.04


,dataset,customers,recency_median,frequency_median,monetary_median,monetary_mean,monetary_total
0,populasi,5852,95.0,3.0,856.02,2916.71,17068582.72
1,sampel,1000,92.0,3.0,892.16,3374.32,3374321.40


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    data = rfm[col]
    sns.histplot(data, bins=50, ax=ax)
    ax.set_title(col)
    if col != "Recency":
        ax.set_xlim(0, data.quantile(0.99))
        ax.set_xlabel(f"{col} (dipotong di p99)")
fig.suptitle("Distribusi RFM sampel 1.000 customer", y=1.02)
fig.tight_layout()
fig.savefig(FIGS / "distribusi_rfm.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Preprocessing KMeans

Recency memakai nilai hari mentah karena rentangnya terbatas. Frequency dan Monetary sangat skewed, sehingga:

1. Di-winsorize pada persentil 99.
2. Ditransformasi dengan `log1p`.
3. Distandarisasi dengan `StandardScaler`.

Dengan cara ini, customer bernilai sangat besar tidak mendominasi pusat cluster.

In [8]:
f_cap = rfm["Frequency"].quantile(0.99)
m_cap = rfm["Monetary"].quantile(0.99)
print(f"Cap Frequency p99: {f_cap:.0f}")
print(f"Cap Monetary p99: {m_cap:,.2f} GBP")

features = pd.DataFrame({
    "Recency": rfm["Recency"],
    "Frequency_log": np.log1p(rfm["Frequency"].clip(upper=f_cap)),
    "Monetary_log": np.log1p(rfm["Monetary"].clip(upper=m_cap)),
})

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
display(features.describe().T)

Cap Frequency p99: 52
Cap Monetary p99: 32,280.03 GBP


,count,mean,std,min,25%,50%,75%,max
Recency,1000.0,201.559000,208.680713,1.000000,25.000000,92.000000,380.000000,739.000000
Frequency_log,1000.0,1.543604,0.793500,0.693147,0.693147,1.386294,2.079442,3.970481
Monetary_log,1000.0,6.770532,1.349857,1.373716,5.813511,6.794753,7.675560,10.382235


## 7. Menetapkan jumlah cluster `k`

Jumlah segmen ditetapkan **3** agar hasil lebih ringkas dan mudah ditindaklanjuti: Champions, Potential, dan Lost. Elbow curve dan silhouette untuk `k=2..10` tetap dihitung sebagai pembanding, tetapi model final memakai `k=3`.

In [9]:
eval_rows = []
for k in range(K_MIN, K_MAX + 1):
    model = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=SEED)
    labels = model.fit_predict(X_scaled)
    eval_rows.append({
        "k": k,
        "inertia": model.inertia_,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df.round(4))

FINAL_K = 3
print(f"k final: {FINAL_K}")

,k,inertia,silhouette,davies_bouldin
0,2,1485.3847,0.4243,0.8819
1,3,971.5549,0.3950,0.8904
2,4,765.9808,0.3547,0.9353
3,5,601.6706,0.3652,0.8946
4,6,517.0609,0.3491,0.9077
5,7,454.4455,0.3409,0.9124
6,8,409.0098,0.3290,0.9566
7,9,365.4862,0.3313,0.9535
8,10,342.1273,0.3310,0.9694


k final: 3


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(eval_df["k"], eval_df["inertia"], marker="o")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(eval_df["k"], eval_df["silhouette"], marker="o", color="darkorange")
axes[1].axvline(FINAL_K, color="red", linestyle="--", label=f"k={FINAL_K}")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / "elbow_silhouette.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Fit KMeans dan profiling segmen

Dengan `k=3`, cluster diberi nama berdasarkan median Monetary dari tinggi ke rendah:

- **Champions**: paling aktif, paling sering, dan paling besar belanja.
- **Potential**: relatif baru, frekuensi dan nilai masih rendah.
- **Lost**: sudah lama tidak belanja dan nilainya kecil.

Nama ini adalah label bisnis; validasi akhir tetap dilakukan dengan profil R/F/M.

In [11]:
final_model = KMeans(
    n_clusters=FINAL_K, init="k-means++", n_init=20, random_state=SEED
)
rfm["Cluster"] = final_model.fit_predict(X_scaled)

profile = rfm.groupby("Cluster").agg(
    n_customers=("CustomerID", "size"),
    R_median=("Recency", "median"),
    F_median=("Frequency", "median"),
    M_median=("Monetary", "median"),
    M_mean=("Monetary", "mean"),
    M_total=("Monetary", "sum"),
    R_score_mean=("R_Score", "mean"),
    F_score_mean=("F_Score", "mean"),
    M_score_mean=("M_Score", "mean"),
    RFM_score_mean=("RFM_Score", "mean"),
    top_country=("Country", lambda s: s.mode().iloc[0] if len(s.mode()) else pd.NA),
)
profile["pct_customers"] = (profile["n_customers"] / len(rfm) * 100).round(2)
profile["revenue_share_pct"] = (profile["M_total"] / rfm["Monetary"].sum() * 100).round(2)

ORDER_BY_VALUE = profile.sort_values("M_median", ascending=False).index.tolist()
NAME_SETS = {
    2: ["Champions", "Lost"],
    3: ["Champions", "Potential", "Lost"],
    4: ["Champions", "Loyal", "Potential", "Lost"],
    5: ["Champions", "Loyal", "At Risk", "Potential", "Lost"],
    6: ["Champions", "Loyal", "At Risk", "Potential", "Hibernating", "Lost"],
}
names = NAME_SETS.get(FINAL_K, [f"Segment {i + 1}" for i in range(FINAL_K)])
name_map = dict(zip(ORDER_BY_VALUE, names[: len(ORDER_BY_VALUE)]))

rfm["Segment"] = rfm["Cluster"].map(name_map)
profile["Segment"] = profile.index.map(name_map)
profile = profile.reset_index()
profile = profile[[
    "Cluster", "Segment", "n_customers", "pct_customers",
    "R_median", "F_median", "M_median", "M_mean", "M_total",
    "revenue_share_pct", "R_score_mean", "F_score_mean",
    "M_score_mean", "RFM_score_mean", "top_country",
]].sort_values("M_median", ascending=False)

display(profile)

rfm.to_csv(DATA / "rfm_clusters.csv", index=False)
profile.to_csv(REPORTS / "cluster_profile.csv", index=False)

,Cluster,Segment,n_customers,pct_customers,R_median,F_median,M_median,M_mean,M_total,revenue_share_pct,R_score_mean,F_score_mean,M_score_mean,RFM_score_mean,top_country
1,1,Champions,305,30.5,26.0,10.0,3444.390,9493.527311,2895525.83,85.81,4.026230,4.639344,4.603279,13.268852,United Kingdom
0,0,Potential,375,37.5,60.0,3.0,679.700,785.120187,294420.07,8.73,3.538667,2.592000,2.589333,8.720000,United Kingdom
2,2,Lost,320,32.0,429.5,1.0,328.105,576.173437,184375.50,5.46,1.390625,1.915625,1.953125,5.259375,United Kingdom


In [12]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(
    data=rfm, x="Recency", y="Monetary", hue="Segment",
    alpha=0.75, ax=ax,
)
ax.set_yscale("log")
ax.set_title("Recency vs Monetary per segmen")
ax.legend(title="Segment", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGS / "cluster_scatter.png", dpi=160, bbox_inches="tight")
plt.show()

In [13]:
heat = profile.set_index("Segment")[["R_median", "F_median", "M_median"]]
heat_z = (heat - heat.mean()) / heat.std(ddof=0)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heat_z, annot=heat.round(1), fmt=".1f",
    cmap="RdYlGn_r", linewidths=0.5, ax=ax,
)
ax.set_title("Profil median RFM per segmen")
fig.tight_layout()
fig.savefig(FIGS / "cluster_profile_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

In [14]:
fig, ax = plt.subplots(figsize=(9, 4))
order = profile.sort_values("revenue_share_pct", ascending=False)
sns.barplot(
    data=order, x="Segment", y="revenue_share_pct",
    hue="Segment", palette="viridis", legend=False, ax=ax,
)
ax.set_title("Kontribusi revenue per segmen")
ax.set_ylabel("Revenue share (%)")
ax.set_xlabel("")
for i, value in enumerate(order["revenue_share_pct"]):
    ax.text(i, value + 0.5, f"{value:.1f}%", ha="center")
fig.tight_layout()
fig.savefig(FIGS / "segment_revenue_share.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Validasi stabilitas

Model dijalankan ulang dengan beberapa seed berbeda. Adjusted Rand Index (ARI) mendekati 1 berarti pembagian cluster stabil dan tidak bergantung pada inisialisasi.

In [15]:
seed_labels = []
for seed in range(5):
    model = KMeans(
        n_clusters=FINAL_K, init="k-means++", n_init=20, random_state=seed
    )
    seed_labels.append(model.fit_predict(X_scaled))

ari_pairs = [
    adjusted_rand_score(seed_labels[i], seed_labels[j])
    for i, j in combinations(range(len(seed_labels)), 2)
]
stability = pd.DataFrame({
    "metric": ["ARI_mean", "ARI_min", "ARI_max"],
    "value": [np.mean(ari_pairs), np.min(ari_pairs), np.max(ari_pairs)],
})
display(stability.round(4))

assert rfm["CustomerID"].is_unique
assert (rfm["Recency"] >= 0).all()
assert (rfm["Frequency"] >= 1).all()
assert (rfm["Monetary"] > 0).all()
assert rfm["Cluster"].notna().all()
assert rfm["Segment"].notna().all()

print("Semua validasi lolos.")

,metric,value
0,ARI_mean,1.0
1,ARI_min,1.0
2,ARI_max,1.0


Semua validasi lolos.


## 10. Ringkasan akhir

Tabel berikut merangkum dataset bersih, sampel, kualitas cluster, dan file yang dihasilkan.

In [16]:
final_summary = pd.DataFrame({
    "metric": [
        "customers_populasi", "customers_sampel", "baris_sampel",
        "invoice_sampel", "snapshot", "k",
        "silhouette", "davies_bouldin", "ari_mean",
    ],
    "value": [
        len(customers), len(rfm), len(sample),
        sample["Invoice"].nunique(), snapshot, FINAL_K,
        round(eval_df.loc[eval_df["k"] == FINAL_K, "silhouette"].iloc[0], 4),
        round(eval_df.loc[eval_df["k"] == FINAL_K, "davies_bouldin"].iloc[0], 4),
        round(np.mean(ari_pairs), 4),
    ],
})
display(final_summary)
final_summary.to_csv(REPORTS / "final_summary.csv", index=False)

print("File yang dihasilkan:")
for folder in (DATA, REPORTS, FIGS):
    for path in sorted(folder.glob("*")):
        print(" -", path)

,metric,value
0,customers_populasi,5852
1,customers_sampel,1000
2,baris_sampel,126269
3,invoice_sampel,6544
4,snapshot,2011-12-10 12:50:00
5,k,3
6,silhouette,0.395
7,davies_bouldin,0.8904
8,ari_mean,1.0


File yang dihasilkan:
 - /Users/march/Documents/MATKUL/PSD/data/clean_transactions_sample.csv
 - /Users/march/Documents/MATKUL/PSD/data/rfm_clusters.csv
 - /Users/march/Documents/MATKUL/PSD/data/rfm_sample.csv
 - /Users/march/Documents/MATKUL/PSD/reports/cleaning_funnel.csv
 - /Users/march/Documents/MATKUL/PSD/reports/cluster_profile.csv
 - /Users/march/Documents/MATKUL/PSD/reports/final_summary.csv
 - /Users/march/Documents/MATKUL/PSD/reports/rfm_kmeans_results.xlsx
 - /Users/march/Documents/MATKUL/PSD/reports/rfm_summary.csv
 - /Users/march/Documents/MATKUL/PSD/figures/cluster_profile_heatmap.png
 - /Users/march/Documents/MATKUL/PSD/figures/cluster_scatter.png
 - /Users/march/Documents/MATKUL/PSD/figures/distribusi_rfm.png
 - /Users/march/Documents/MATKUL/PSD/figures/elbow_silhouette.png
 - /Users/march/Documents/MATKUL/PSD/figures/segment_revenue_share.png


## 11. Export ke Excel

Semua hasil analisis disimpan ke satu workbook yang rapi:

`reports/rfm_kmeans_results.xlsx`

Workbook berisi:

- `Dashboard` — metrik utama, tabel segmen, dan gambar grafik
- `Ringkasan`
- `Cleaning_Funnel`
- `RFM_Summary`
- `Evaluasi_k`
- `Cluster_Profile`
- `RFM_Customers`
- `Sample_Transaksi`
- `Stabilitas`
- `Rumus` — dokumentasi formula Excel dan matematis
- `Perhitungan_RFM` — formula Excel asli per customer
- `Mapping_Segmen` — mapping cluster ke nama segmen
- `KMeans_Centroids` — centroid dalam skala terstandarisasi
- `Parameter` — snapshot, cap outlier, mean, dan std
- `Invoice_Unik` — daftar invoice unik per customer

Formatting yang diterapkan: header bold, freeze pane, autofilter, lebar kolom otomatis, format mata uang GBP, format persentase, conditional formatting, dan `fullCalcOnLoad` agar Excel menghitung ulang formula saat file dibuka.

In [17]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule

EXCEL_PATH = REPORTS / "rfm_kmeans_results.xlsx"
if EXCEL_PATH.exists():
    EXCEL_PATH.unlink()

# ---------------------------------------------------------------------------
# Data untuk sheet Excel
# ---------------------------------------------------------------------------
profile_x = profile.copy()
profile_x["pct_customers"] = profile_x["pct_customers"] / 100
profile_x["revenue_share_pct"] = profile_x["revenue_share_pct"] / 100

rfm_x = rfm.copy()
sample_x = sample[[
    "Invoice", "StockCode", "Description", "Quantity", "InvoiceDate",
    "UnitPrice", "CustomerID", "Country", "Revenue", "SourceSheet",
]].copy()

invoice_unique = (
    sample[["CustomerID", "Invoice"]]
    .drop_duplicates()
    .sort_values(["CustomerID", "Invoice"])
    .reset_index(drop=True)
)

mapping_segmen = pd.DataFrame({
    "Cluster": list(name_map.keys()),
    "Segment": list(name_map.values()),
}).sort_values("Cluster").reset_index(drop=True)

centroids = pd.DataFrame(
    final_model.cluster_centers_,
    columns=["R_scaled", "F_scaled", "M_scaled"],
)
centroids.insert(0, "Cluster", range(FINAL_K))

parameters = pd.DataFrame({
    "Parameter": [
        "Snapshot", "Cap_Frequency_p99", "Cap_Monetary_p99",
        "Mean_R", "Std_R", "Mean_F_log", "Std_F_log",
        "Mean_M_log", "Std_M_log",
    ],
    "Nilai": [
        snapshot, f_cap, m_cap,
        features["Recency"].mean(), features["Recency"].std(ddof=0),
        features["Frequency_log"].mean(), features["Frequency_log"].std(ddof=0),
        features["Monetary_log"].mean(), features["Monetary_log"].std(ddof=0),
    ],
    "Keterangan": [
        "Tanggal maksimum dataset + 1 hari",
        "Persentil 99 Frequency",
        "Persentil 99 Monetary",
        "Rata-rata Recency untuk StandardScaler",
        "Std populasi Recency",
        "Rata-rata log1p(Frequency)",
        "Std populasi log1p(Frequency)",
        "Rata-rata log1p(Monetary)",
        "Std populasi log1p(Monetary)",
    ],
})

# ---------------------------------------------------------------------------
# Sheet Perhitungan_RFM: setiap sel berisi formula Excel asli
# ---------------------------------------------------------------------------
last_sample_row = len(sample_x) + 1
last_invoice_row = len(invoice_unique) + 1
last_rfm_row = len(rfm_x) + 1

rumus_perhitungan = []
for row, customer_id in enumerate(rfm_x["CustomerID"], start=2):
    rumus_perhitungan.append({
        "CustomerID": customer_id,
        "LastPurchase": (
            f"=SUMPRODUCT(MAX((Sample_Transaksi!$G$2:$G${last_sample_row}=$A{row})"
            f"*Sample_Transaksi!$E$2:$E${last_sample_row}))"
        ),
        "Recency": f"=INT(Parameter!$B$2-B{row})",
        "Frequency": f"=COUNTIFS(Invoice_Unik!$A$2:$A${last_invoice_row},$A{row})",
        "Monetary": (
            f"=SUMIFS(Sample_Transaksi!$I$2:$I${last_sample_row},"
            f"Sample_Transaksi!$G$2:$G${last_sample_row},$A{row})"
        ),
        "Frequency_cap": f"=MIN(D{row},Parameter!$B$3)",
        "Monetary_cap": f"=MIN(E{row},Parameter!$B$4)",
        "R_scaled": f"=(C{row}-Parameter!$B$5)/Parameter!$B$6",
        "F_scaled": f"=(LN(1+F{row})-Parameter!$B$7)/Parameter!$B$8",
        "M_scaled": f"=(LN(1+G{row})-Parameter!$B$9)/Parameter!$B$10",
        "Jarak_Cluster_0": (
            f"=SQRT((H{row}-KMeans_Centroids!$B$2)^2"
            f"+(I{row}-KMeans_Centroids!$C$2)^2"
            f"+(J{row}-KMeans_Centroids!$D$2)^2)"
        ),
        "Jarak_Cluster_1": (
            f"=SQRT((H{row}-KMeans_Centroids!$B$3)^2"
            f"+(I{row}-KMeans_Centroids!$C$3)^2"
            f"+(J{row}-KMeans_Centroids!$D$3)^2)"
        ),
        "Jarak_Cluster_2": (
            f"=SQRT((H{row}-KMeans_Centroids!$B$4)^2"
            f"+(I{row}-KMeans_Centroids!$C$4)^2"
            f"+(J{row}-KMeans_Centroids!$D$4)^2)"
        ),
        "Cluster": f"=IF(AND(K{row}<=L{row},K{row}<=M{row}),0,IF(L{row}<=M{row},1,2))",
        "Segment": f"=VLOOKUP(N{row},Mapping_Segmen!$A$2:$B${FINAL_K + 1},2,FALSE)",
        "R_Score": (
            f"=1+INT((RANK(C{row},$C$2:$C${last_rfm_row},0)"
            f"+COUNTIF($C$2:C{row},C{row})-2)/(COUNT($C$2:$C${last_rfm_row})/5))"
        ),
        "F_Score": (
            f"=1+INT((RANK(D{row},$D$2:$D${last_rfm_row},1)"
            f"+COUNTIF($D$2:D{row},D{row})-2)/(COUNT($D$2:$D${last_rfm_row})/5))"
        ),
        "M_Score": (
            f"=1+INT((RANK(E{row},$E$2:$E${last_rfm_row},1)"
            f"+COUNTIF($E$2:E{row},E{row})-2)/(COUNT($E$2:$E${last_rfm_row})/5))"
        ),
        "RFM_Score": f"=P{row}+Q{row}+R{row}",
        "RFM_Cell": f"=P{row}&Q{row}&R{row}",
    })

perhitungan = pd.DataFrame(rumus_perhitungan)

# ---------------------------------------------------------------------------
# Tabel dokumentasi rumus
# ---------------------------------------------------------------------------
rumus_rows = [
    {
        "No": 1, "Tahap": "Revenue", "Komponen": "Revenue per baris",
        "Rumus Excel": "=D2*F2",
        "Rumus Matematis": "Revenue = Quantity x UnitPrice",
        "Penjelasan": "Nilai penjualan tiap baris transaksi.",
        "Contoh di Sheet": "Sample_Transaksi!I2",
    },
    {
        "No": 2, "Tahap": "RFM", "Komponen": "Snapshot",
        "Rumus Excel": "=MAX(Sample_Transaksi!E:E)+1",
        "Rumus Matematis": "Snapshot = max(InvoiceDate) + 1 hari",
        "Penjelasan": "Tanggal acuan untuk Recency.",
        "Contoh di Sheet": "Parameter!B2",
    },
    {
        "No": 3, "Tahap": "RFM", "Komponen": "Last purchase",
        "Rumus Excel": (
            f"=SUMPRODUCT(MAX((Sample_Transaksi!$G$2:$G${last_sample_row}=$A2)"
            f"*Sample_Transaksi!$E$2:$E${last_sample_row}))"
        ),
        "Rumus Matematis": "LastPurchase = max(InvoiceDate) per customer",
        "Penjelasan": "Mengambil tanggal transaksi terakhir customer.",
        "Contoh di Sheet": "Perhitungan_RFM!B2",
    },
    {
        "No": 4, "Tahap": "RFM", "Komponen": "Recency",
        "Rumus Excel": "=INT(Parameter!$B$2-B2)",
        "Rumus Matematis": "R = snapshot - last purchase",
        "Penjelasan": "Jumlah hari sejak transaksi terakhir.",
        "Contoh di Sheet": "Perhitungan_RFM!C2",
    },
    {
        "No": 5, "Tahap": "RFM", "Komponen": "Frequency",
        "Rumus Excel": f"=COUNTIFS(Invoice_Unik!$A$2:$A${last_invoice_row},$A2)",
        "Rumus Matematis": "F = jumlah invoice unik",
        "Penjelasan": "Menghitung invoice unik per customer dari sheet Invoice_Unik.",
        "Contoh di Sheet": "Perhitungan_RFM!D2",
    },
    {
        "No": 6, "Tahap": "RFM", "Komponen": "Monetary",
        "Rumus Excel": (
            f"=SUMIFS(Sample_Transaksi!$I$2:$I${last_sample_row},"
            f"Sample_Transaksi!$G$2:$G${last_sample_row},$A2)"
        ),
        "Rumus Matematis": "M = sum(Revenue)",
        "Penjelasan": "Total belanja customer.",
        "Contoh di Sheet": "Perhitungan_RFM!E2",
    },
    {
        "No": 7, "Tahap": "Preprocessing", "Komponen": "Winsorize Frequency",
        "Rumus Excel": "=MIN(D2,Parameter!$B$3)",
        "Rumus Matematis": "F_cap = min(F, p99(F))",
        "Penjelasan": "Membatasi outlier Frequency.",
        "Contoh di Sheet": "Perhitungan_RFM!F2",
    },
    {
        "No": 8, "Tahap": "Preprocessing", "Komponen": "Winsorize Monetary",
        "Rumus Excel": "=MIN(E2,Parameter!$B$4)",
        "Rumus Matematis": "M_cap = min(M, p99(M))",
        "Penjelasan": "Membatasi outlier Monetary.",
        "Contoh di Sheet": "Perhitungan_RFM!G2",
    },
    {
        "No": 9, "Tahap": "KMeans", "Komponen": "R scaled",
        "Rumus Excel": "=(C2-Parameter!$B$5)/Parameter!$B$6",
        "Rumus Matematis": "(R - mean_R) / std_R",
        "Penjelasan": "StandardScaler untuk Recency.",
        "Contoh di Sheet": "Perhitungan_RFM!H2",
    },
    {
        "No": 10, "Tahap": "KMeans", "Komponen": "F scaled",
        "Rumus Excel": "=(LN(1+F2)-Parameter!$B$7)/Parameter!$B$8",
        "Rumus Matematis": "(ln(1+F_cap) - mean_F) / std_F",
        "Penjelasan": "Log1p lalu standardisasi Frequency.",
        "Contoh di Sheet": "Perhitungan_RFM!I2",
    },
    {
        "No": 11, "Tahap": "KMeans", "Komponen": "M scaled",
        "Rumus Excel": "=(LN(1+G2)-Parameter!$B$9)/Parameter!$B$10",
        "Rumus Matematis": "(ln(1+M_cap) - mean_M) / std_M",
        "Penjelasan": "Log1p lalu standardisasi Monetary.",
        "Contoh di Sheet": "Perhitungan_RFM!J2",
    },
    {
        "No": 12, "Tahap": "KMeans", "Komponen": "Jarak ke cluster 0",
        "Rumus Excel": (
            "=SQRT((H2-KMeans_Centroids!$B$2)^2"
            "+(I2-KMeans_Centroids!$C$2)^2"
            "+(J2-KMeans_Centroids!$D$2)^2)"
        ),
        "Rumus Matematis": "d = sqrt(sum((x - centroid)^2))",
        "Penjelasan": "Jarak Euclidean ke centroid cluster 0.",
        "Contoh di Sheet": "Perhitungan_RFM!K2",
    },
    {
        "No": 13, "Tahap": "KMeans", "Komponen": "Jarak ke cluster 1",
        "Rumus Excel": (
            "=SQRT((H2-KMeans_Centroids!$B$3)^2"
            "+(I2-KMeans_Centroids!$C$3)^2"
            "+(J2-KMeans_Centroids!$D$3)^2)"
        ),
        "Rumus Matematis": "d = sqrt(sum((x - centroid)^2))",
        "Penjelasan": "Jarak Euclidean ke centroid cluster 1.",
        "Contoh di Sheet": "Perhitungan_RFM!L2",
    },
    {
        "No": 14, "Tahap": "KMeans", "Komponen": "Jarak ke cluster 2",
        "Rumus Excel": (
            "=SQRT((H2-KMeans_Centroids!$B$4)^2"
            "+(I2-KMeans_Centroids!$C$4)^2"
            "+(J2-KMeans_Centroids!$D$4)^2)"
        ),
        "Rumus Matematis": "d = sqrt(sum((x - centroid)^2))",
        "Penjelasan": "Jarak Euclidean ke centroid cluster 2.",
        "Contoh di Sheet": "Perhitungan_RFM!M2",
    },
    {
        "No": 15, "Tahap": "KMeans", "Komponen": "Cluster",
        "Rumus Excel": "=IF(AND(K2<=L2,K2<=M2),0,IF(L2<=M2,1,2))",
        "Rumus Matematis": "cluster = argmin(distance)",
        "Penjelasan": "Memilih centroid dengan jarak terkecil.",
        "Contoh di Sheet": "Perhitungan_RFM!N2",
    },
    {
        "No": 16, "Tahap": "Segmentasi", "Komponen": "Segment",
        "Rumus Excel": f"=VLOOKUP(N2,Mapping_Segmen!$A$2:$B${FINAL_K + 1},2,FALSE)",
        "Rumus Matematis": "segment = mapping[cluster]",
        "Penjelasan": "Mengubah nomor cluster menjadi nama segmen.",
        "Contoh di Sheet": "Perhitungan_RFM!O2",
    },
    {
        "No": 17, "Tahap": "RFM Score", "Komponen": "R score",
        "Rumus Excel": (
            f"=1+INT((RANK(C2,$C$2:$C${last_rfm_row},0)"
            f"+COUNTIF($C$2:C2,C2)-2)/(COUNT($C$2:$C${last_rfm_row})/5))"
        ),
        "Rumus Matematis": "Skor R 1-5 dengan tie-breaker urutan baris",
        "Penjelasan": "Recency kecil mendapat skor besar; COUNTIF memecah nilai seri agar kuintil sama dengan qcut Python.",
        "Contoh di Sheet": "Perhitungan_RFM!P2",
    },
    {
        "No": 18, "Tahap": "RFM Score", "Komponen": "F score",
        "Rumus Excel": (
            f"=1+INT((RANK(D2,$D$2:$D${last_rfm_row},1)"
            f"+COUNTIF($D$2:D2,D2)-2)/(COUNT($D$2:$D${last_rfm_row})/5))"
        ),
        "Rumus Matematis": "Skor F 1-5 dengan tie-breaker urutan baris",
        "Penjelasan": "Frequency besar mendapat skor besar; COUNTIF memecah nilai seri agar kuintil sama dengan qcut Python.",
        "Contoh di Sheet": "Perhitungan_RFM!Q2",
    },
    {
        "No": 19, "Tahap": "RFM Score", "Komponen": "M score",
        "Rumus Excel": (
            f"=1+INT((RANK(E2,$E$2:$E${last_rfm_row},1)"
            f"+COUNTIF($E$2:E2,E2)-2)/(COUNT($E$2:$E${last_rfm_row})/5))"
        ),
        "Rumus Matematis": "Skor M 1-5 dengan tie-breaker urutan baris",
        "Penjelasan": "Monetary besar mendapat skor besar; COUNTIF memecah nilai seri agar kuintil sama dengan qcut Python.",
        "Contoh di Sheet": "Perhitungan_RFM!R2",
    },
    {
        "No": 20, "Tahap": "RFM Score", "Komponen": "RFM score",
        "Rumus Excel": "=P2+Q2+R2",
        "Rumus Matematis": "RFM = R_score + F_score + M_score",
        "Penjelasan": "Total skor RFM.",
        "Contoh di Sheet": "Perhitungan_RFM!S2",
    },
    {
        "No": 21, "Tahap": "RFM Score", "Komponen": "RFM cell",
        "Rumus Excel": "=P2&Q2&R2",
        "Rumus Matematis": "Gabungan skor R, F, M",
        "Penjelasan": "Contoh: 335.",
        "Contoh di Sheet": "Perhitungan_RFM!T2",
    },
    {
        "No": 22, "Tahap": "Profil Cluster", "Komponen": "Jumlah customer",
        "Rumus Excel": f"=COUNTIF(RFM_Customers!$K$2:$K${last_rfm_row},A2)",
        "Rumus Matematis": "n = count(cluster)",
        "Penjelasan": "Jumlah customer per cluster.",
        "Contoh di Sheet": "Cluster_Profile!C2",
    },
    {
        "No": 23, "Tahap": "Profil Cluster", "Komponen": "Revenue share",
        "Rumus Excel": (
            f"=SUMIFS(RFM_Customers!$D$2:$D${last_rfm_row},"
            f"RFM_Customers!$K$2:$K${last_rfm_row},A2)"
            f"/SUM(RFM_Customers!$D$2:$D${last_rfm_row})"
        ),
        "Rumus Matematis": "share = sum(M_cluster) / sum(M)",
        "Penjelasan": "Kontribusi revenue tiap cluster.",
        "Contoh di Sheet": "Cluster_Profile!J2",
    },
]

# ---------------------------------------------------------------------------
# Styling helpers
# ---------------------------------------------------------------------------
HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
THIN = Side(style="thin", color="D9E2F3")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)


def style_header(ws):
    for cell in ws[1]:
        if cell.value is not None:
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = BORDER
    ws.freeze_panes = "A2"
    if ws.max_row > 1:
        ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 24


def set_widths(ws, widths, max_scan=300):
    if widths:
        for col, width in widths.items():
            ws.column_dimensions[col].width = width
        return
    for col_idx, col_cells in enumerate(
        ws.iter_cols(min_row=1, max_row=min(ws.max_row, max_scan)), start=1
    ):
        max_len = 0
        for cell in col_cells:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[get_column_letter(col_idx)].width = min(
            max(10, max_len + 2), 50
        )


def format_sheet(ws, number_formats=None, widths=None, max_scan=300):
    style_header(ws)
    set_widths(ws, widths, max_scan=max_scan)
    if number_formats:
        headers = {cell.value: cell.column for cell in ws[1] if cell.value is not None}
        for header, fmt in number_formats.items():
            col_idx = headers.get(header)
            if col_idx is None:
                continue
            for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    cell.number_format = fmt


# ---------------------------------------------------------------------------
# Tulis workbook
# ---------------------------------------------------------------------------
with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    final_summary.to_excel(writer, sheet_name="Ringkasan", index=False)
    funnel_df.to_excel(writer, sheet_name="Cleaning_Funnel", index=False)
    rfm_summary.to_excel(writer, sheet_name="RFM_Summary", index=False)
    eval_df.to_excel(writer, sheet_name="Evaluasi_k", index=False)
    profile_x.to_excel(writer, sheet_name="Cluster_Profile", index=False)
    rfm_x.to_excel(writer, sheet_name="RFM_Customers", index=False)
    sample_x.to_excel(writer, sheet_name="Sample_Transaksi", index=False)
    stability.to_excel(writer, sheet_name="Stabilitas", index=False)
    perhitungan.to_excel(writer, sheet_name="Perhitungan_RFM", index=False)
    mapping_segmen.to_excel(writer, sheet_name="Mapping_Segmen", index=False)
    centroids.to_excel(writer, sheet_name="KMeans_Centroids", index=False)
    parameters.to_excel(writer, sheet_name="Parameter", index=False)
    invoice_unique.to_excel(writer, sheet_name="Invoice_Unik", index=False)

    wb = writer.book

    format_sheet(wb["Ringkasan"])
    format_sheet(
        wb["Cleaning_Funnel"],
        number_formats={"before": "#,##0", "removed": "#,##0", "remaining": "#,##0"},
    )
    format_sheet(
        wb["RFM_Summary"],
        number_formats={
            "customers": "#,##0",
            "recency_median": "#,##0",
            "frequency_median": "#,##0",
            "monetary_median": "£#,##0.00",
            "monetary_mean": "£#,##0.00",
            "monetary_total": "£#,##0.00",
        },
    )
    format_sheet(
        wb["Evaluasi_k"],
        number_formats={
            "k": "0",
            "inertia": "#,##0.00",
            "silhouette": "0.0000",
            "davies_bouldin": "0.0000",
        },
    )
    format_sheet(
        wb["Cluster_Profile"],
        number_formats={
            "Cluster": "0",
            "n_customers": "#,##0",
            "pct_customers": "0.00%",
            "R_median": "#,##0",
            "F_median": "#,##0",
            "M_median": "£#,##0.00",
            "M_mean": "£#,##0.00",
            "M_total": "£#,##0.00",
            "revenue_share_pct": "0.00%",
            "R_score_mean": "0.00",
            "F_score_mean": "0.00",
            "M_score_mean": "0.00",
            "RFM_score_mean": "0.00",
        },
    )
    format_sheet(
        wb["RFM_Customers"],
        number_formats={
            "Recency": "#,##0",
            "Frequency": "#,##0",
            "Monetary": "£#,##0.00",
            "R_Score": "0",
            "F_Score": "0",
            "M_Score": "0",
            "RFM_Score": "0",
            "Cluster": "0",
        },
    )
    format_sheet(
        wb["Sample_Transaksi"],
        number_formats={
            "Quantity": "#,##0",
            "UnitPrice": "£#,##0.00",
            "Revenue": "£#,##0.00",
            "InvoiceDate": "yyyy-mm-dd hh:mm",
        },
        widths={
            "A": 12, "B": 12, "C": 48, "D": 10,
            "E": 19, "F": 11, "G": 12, "H": 20,
            "I": 14, "J": 18,
        },
    )
    format_sheet(wb["Stabilitas"], number_formats={"value": "0.0000"})
    format_sheet(
        wb["Perhitungan_RFM"],
        number_formats={
            "LastPurchase": "yyyy-mm-dd hh:mm",
            "Recency": "#,##0",
            "Frequency": "#,##0",
            "Monetary": "£#,##0.00",
            "Frequency_cap": "#,##0.00",
            "Monetary_cap": "£#,##0.00",
            "R_scaled": "0.0000",
            "F_scaled": "0.0000",
            "M_scaled": "0.0000",
            "Jarak_Cluster_0": "0.0000",
            "Jarak_Cluster_1": "0.0000",
            "Jarak_Cluster_2": "0.0000",
            "Cluster": "0",
            "R_Score": "0",
            "F_Score": "0",
            "M_Score": "0",
            "RFM_Score": "0",
        },
    )
    format_sheet(
        wb["Mapping_Segmen"],
        widths={"A": 10, "B": 16, "C": 55},
    )
    format_sheet(
        wb["KMeans_Centroids"],
        number_formats={
            "Cluster": "0",
            "R_scaled": "0.0000",
            "F_scaled": "0.0000",
            "M_scaled": "0.0000",
        },
    )
    format_sheet(
        wb["Parameter"],
        number_formats={"Nilai": "0.0000"},
        widths={"A": 24, "B": 20, "C": 52},
    )
    wb["Parameter"]["B2"].number_format = "yyyy-mm-dd hh:mm"
    wb["Parameter"]["B3"].number_format = "#,##0.00"
    wb["Parameter"]["B4"].number_format = "£#,##0.00"
    format_sheet(
        wb["Invoice_Unik"],
        number_formats={"Jumlah_Baris": "#,##0"},
        widths={"A": 12, "B": 12, "C": 14},
    )

    # Conditional formatting
    ws_rfm = wb["RFM_Customers"]
    headers = {cell.value: cell.column for cell in ws_rfm[1] if cell.value is not None}
    mon_col = get_column_letter(headers["Monetary"])
    ws_rfm.conditional_formatting.add(
        f"{mon_col}2:{mon_col}{ws_rfm.max_row}",
        ColorScaleRule(
            start_type="min", start_color="FFFFFF",
            mid_type="percentile", mid_value=50, mid_color="FFEB84",
            end_type="max", end_color="F8696B",
        ),
    )

    ws_prof = wb["Cluster_Profile"]
    prof_headers = {cell.value: cell.column for cell in ws_prof[1] if cell.value is not None}
    rev_col = get_column_letter(prof_headers["revenue_share_pct"])
    ws_prof.conditional_formatting.add(
        f"{rev_col}2:{rev_col}{ws_prof.max_row}",
        DataBarRule(
            start_type="num", start_value=0,
            end_type="num", end_value=1, color="5B9BD5",
        ),
    )

    # Sheet Rumus
    ws_rumus = wb.create_sheet("Rumus", 8)
    rumus_headers = [
        "No", "Tahap", "Komponen", "Rumus Excel",
        "Rumus Matematis", "Penjelasan", "Contoh di Sheet",
    ]
    for col, header in enumerate(rumus_headers, start=1):
        cell = ws_rumus.cell(row=1, column=col, value=header)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BORDER
    for i, row in enumerate(rumus_rows, start=2):
        for j, header in enumerate(rumus_headers, start=1):
            value = row[header]
            cell = ws_rumus.cell(row=i, column=j, value=value)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = BORDER
            if header == "Rumus Excel" and isinstance(value, str) and value.startswith("="):
                cell.data_type = "s"
    ws_rumus.freeze_panes = "A2"
    ws_rumus.auto_filter.ref = ws_rumus.dimensions
    ws_rumus.column_dimensions["A"].width = 6
    ws_rumus.column_dimensions["B"].width = 18
    ws_rumus.column_dimensions["C"].width = 24
    ws_rumus.column_dimensions["D"].width = 75
    ws_rumus.column_dimensions["E"].width = 42
    ws_rumus.column_dimensions["F"].width = 48
    ws_rumus.column_dimensions["G"].width = 22
    ws_rumus.row_dimensions[1].height = 24

    note_row = len(rumus_rows) + 3
    ws_rumus.cell(
        row=note_row, column=1,
        value=(
            "Catatan: sheet Perhitungan_RFM berisi formula Excel asli untuk setiap customer. "
            "Sheet RFM_Customers berisi nilai hasil Python. Skor memakai RANK + COUNTIF sebagai "
            "tie-breaker agar pembagian kuintil sama dengan qcut Python (rank method='first')."
        ),
    )
    ws_rumus.merge_cells(
        start_row=note_row, start_column=1, end_row=note_row, end_column=7
    )
    ws_rumus.cell(row=note_row, column=1).alignment = Alignment(wrap_text=True, vertical="top")
    ws_rumus.row_dimensions[note_row].height = 42

    # Dashboard
    ws = wb.create_sheet("Dashboard", 0)
    ws.sheet_view.showGridLines = False
    ws.sheet_view.zoomScale = 90

    ws.merge_cells("A1:G1")
    ws["A1"] = "RFM & KMeans — Online Retail II"
    ws["A1"].font = Font(bold=True, size=20, color="1F4E78")
    ws["A1"].alignment = Alignment(vertical="center")
    ws.row_dimensions[1].height = 30

    ws.merge_cells("A2:G2")
    ws["A2"] = (
        f"Snapshot {snapshot:%Y-%m-%d}  |  "
        f"Sampel {len(rfm):,} customer  |  "
        f"{len(sample):,} transaksi  |  "
        f"Retur/pembatalan dikeluarkan  |  {FINAL_K} segmen"
    )
    ws["A2"].font = Font(size=11, color="595959")

    ws["A4"] = "Metrik"
    ws["B4"] = "Nilai"
    for cell in (ws["A4"], ws["B4"]):
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center")
        cell.border = BORDER

    sil_value = float(eval_df.loc[eval_df["k"] == FINAL_K, "silhouette"].iloc[0])
    ari_value = float(np.mean(ari_pairs))
    metrics = [
        ("Customer sampel", len(rfm), "#,##0"),
        ("Transaksi sampel", len(sample), "#,##0"),
        ("Invoice sampel", sample["Invoice"].nunique(), "#,##0"),
        ("Jumlah cluster (k)", FINAL_K, "0"),
        ("Silhouette score", sil_value, "0.0000"),
        ("ARI stabilitas", ari_value, "0.0000"),
    ]
    for i, (label, value, fmt) in enumerate(metrics, start=5):
        c_label = ws.cell(row=i, column=1, value=label)
        c_label.font = Font(bold=True)
        c_label.border = BORDER
        c_value = ws.cell(row=i, column=2, value=value)
        c_value.number_format = fmt
        c_value.border = BORDER

    ws["A12"] = "Segmen"
    ws["B12"] = "Customer"
    ws["C12"] = "% Customer"
    ws["D12"] = "Revenue Share"
    ws["E12"] = "Median R"
    ws["F12"] = "Median F"
    ws["G12"] = "Median M"
    for col in "ABCDEFG":
        cell = ws[f"{col}12"]
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = BORDER

    seg = profile_x.sort_values("M_median", ascending=False)
    for i, row in enumerate(seg.itertuples(index=False), start=13):
        values = [
            row.Segment, row.n_customers, row.pct_customers,
            row.revenue_share_pct, row.R_median, row.F_median, row.M_median,
        ]
        for j, value in enumerate(values, start=1):
            cell = ws.cell(row=i, column=j, value=value)
            cell.border = BORDER
            if j == 1:
                cell.font = Font(bold=True)
            elif j == 2:
                cell.number_format = "#,##0"
            elif j in (3, 4):
                cell.number_format = "0.00%"
            elif j in (5, 6):
                cell.number_format = "#,##0"
            elif j == 7:
                cell.number_format = "£#,##0.00"

    for col, width in {
        "A": 26, "B": 12, "C": 14, "D": 16,
        "E": 12, "F": 12, "G": 16, "H": 2,
        "I": 18, "J": 18, "K": 18, "L": 18,
    }.items():
        ws.column_dimensions[col].width = width

    def add_image(anchor, filename, width, height):
        path = FIGS / filename
        if not path.exists():
            return
        img = XLImage(str(path))
        img.width = width
        img.height = height
        ws.add_image(img, anchor)

    captions = [
        (21, "Kontribusi revenue per segmen"),
        (21, "Recency vs Monetary per segmen", "I"),
        (40, "Elbow method dan silhouette score"),
        (42, "Profil median RFM per segmen", "I"),
        (57, "Distribusi RFM sampel"),
    ]
    for item in captions:
        row = item[0]
        text = item[1]
        col = item[2] if len(item) > 2 else "A"
        ws[f"{col}{row}"] = text
        ws[f"{col}{row}"].font = Font(bold=True, size=12, color="1F4E78")

    add_image("A22", "segment_revenue_share.png", 620, 280)
    add_image("I22", "cluster_scatter.png", 620, 380)
    add_image("A41", "elbow_silhouette.png", 700, 260)
    add_image("I43", "cluster_profile_heatmap.png", 560, 350)
    add_image("A58", "distribusi_rfm.png", 760, 240)

    ws["A75"] = (
        "Sumber: online_retail_II.xlsx. RFM memakai transaksi pembelian valid; "
        "retur, cancellation, adjustment, dan customer tanpa ID dikeluarkan. "
        "Rumus lengkap ada di sheet Rumus dan Perhitungan_RFM."
    )
    ws["A75"].font = Font(size=9, italic=True, color="7F7F7F")

    wb.active = 0
    wb.calculation.calcMode = "auto"
    wb.calculation.fullCalcOnLoad = True
    wb.calculation.forceFullCalc = True

# ---------------------------------------------------------------------------
# Validasi workbook
# ---------------------------------------------------------------------------
check = load_workbook(EXCEL_PATH, read_only=True, data_only=False)
expected_sheets = [
    "Dashboard", "Ringkasan", "Cleaning_Funnel", "RFM_Summary",
    "Evaluasi_k", "Cluster_Profile", "RFM_Customers", "Sample_Transaksi",
    "Stabilitas", "Rumus", "Perhitungan_RFM", "Mapping_Segmen",
    "KMeans_Centroids", "Parameter", "Invoice_Unik",
]
actual_sheets = check.sheetnames
print("Sheets:", actual_sheets)
assert actual_sheets == expected_sheets
assert check["RFM_Customers"].max_row == len(rfm) + 1
assert check["Sample_Transaksi"].max_row == len(sample) + 1
assert check["Cluster_Profile"].max_row == len(profile_x) + 1
assert check["Perhitungan_RFM"].max_row == len(rfm) + 1
assert check["KMeans_Centroids"].max_row == FINAL_K + 1
assert check["Mapping_Segmen"].max_row == FINAL_K + 1
assert check["Invoice_Unik"].max_row == len(invoice_unique) + 1
assert check["Perhitungan_RFM"]["B2"].value.startswith("=")
assert check["Perhitungan_RFM"]["C2"].value.startswith("=")
assert check["Perhitungan_RFM"]["N2"].value.startswith("=")
assert check["Rumus"]["D2"].data_type == "s"
check.close()

print(f"Jumlah segmen: {FINAL_K}")
print(f"Excel tersimpan: {EXCEL_PATH}")
print(f"Ukuran file: {EXCEL_PATH.stat().st_size / 1024 / 1024:.1f} MB")


Sheets: ['Dashboard', 'Ringkasan', 'Cleaning_Funnel', 'RFM_Summary', 'Evaluasi_k', 'Cluster_Profile', 'RFM_Customers', 'Sample_Transaksi', 'Stabilitas', 'Rumus', 'Perhitungan_RFM', 'Mapping_Segmen', 'KMeans_Centroids', 'Parameter', 'Invoice_Unik']
Jumlah segmen: 3
Excel tersimpan: /Users/march/Documents/MATKUL/PSD/reports/rfm_kmeans_results.xlsx
Ukuran file: 7.8 MB


## Interpretasi

- Segmen **Champions** biasanya menyumbang revenue terbesar meskipun jumlah customer-nya kecil. Prioritas: retensi, akses produk baru, dan layanan premium.
- Segmen **Potential** perlu didorong frekuensi dan nilai belanjanya agar naik ke Champions.
- Segmen **Lost** sebaiknya diberi kampanye win-back berbiaya rendah.

Karena sampel diambil acak per customer, hasil ini dapat dijadikan estimasi untuk populasi 5.852 customer. Untuk keputusan operasional, ulangi analisis pada seluruh populasi atau validasi dengan periode yang lebih baru.